In [42]:
import numpy as np
import sympy as sym
from sympy import pprint
#Rough
def pp(a, b = 'Full'):
    if b == 'Full':
        c = sym.Matrix(a)
    else:
        c = np.round(a.real, b) + 1j*np.round(a.imag, b)
        c = sym.Matrix(c)
    pprint(c)


In [32]:
class QuantumSystem:
    def __init__(self, q, alpha = 'auto'):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        self.alpha = alpha

        self.psi0_a = self.superposition_state()
        self.zero_b = self.superposition_state(coeffs = [1] + [0]*(self.N-1))
        self.psi_kron_0 = np.kron(self.psi0_a, self.zero_b)
        
        self.I_a = np.eye(self.N)
        self.I_b = np.eye(self.N)
        self.Proj_ab = np.kron(self.I_a, self.zero_b @ self.zero_b.conj().T)    # projection operator - Identity on a, projection on |0> on b
        
        self.Phase = np.diag(self.phase_operator_phases(alpha))                      # phase operator from the phases, "U_a" in notes
        self.Swap = self.U_ab()                                                 # swap operator, "U_ab" in notes
        
    def matrix_basis(self):
        q = self.q
        N = 2**q
        basis = []
        for i in range(N):
            state_i = np.zeros(N)
            state_i[i] = 1
            basis.append(state_i)
        return np.array(basis)

    def superposition_state(self, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
        q = self.q
        N = self.N
        if coeffs == 'maximal_except_0':
            coeffs = np.ones(N)
            coeffs[0] = 0

        basis = self.matrix_basis()

        c = np.array(coeffs).reshape(1, N)
        c = c/np.sqrt(np.sum(np.abs(c)**2))  # normalizing 

        if basis.shape[1] != c.shape[1]:
            raise ValueError('Error: Shapes of coefficients and basis do not match')
        if len(coeffs) != N:            # checking length
            raise ValueError('Error: The number of coefficients is not 2^q')

        sup_state = np.sum(c * basis, axis = 1, keepdims = True)
        if (np.linalg.norm(sup_state)-1) > 1e-10:
            raise ValueError('Error: The resultant state is not normalized')
        return sup_state
    
    
    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U


    def phase_operator_phases(self, alpha = 'auto'):       # phases for the phase operator
        q, N = self.q, self.N
        # alpha is the coefficient in the Hamiltonian
        if q%2 == 0:            # creating alpha automatically, see notebook
            alpha = 4*np.pi/q**2
        else:
            alpha = 4*np.pi/(q**2-1)
        if alpha != 'auto':
            alpha = alpha           

        def qubit_sequence(a,q):
            b = np.binary_repr(a, width = q)       # string
            c = np.array([int(i) for i in b])   # array
            return c

        def edges_count(seq):   #seq is a binary sequence array
            p_0 = np.count_nonzero(seq == 0)
            p_1 = np.count_nonzero(seq == 1)
            return int(p_0*p_1)

        phases = []
        for i in range(N):
            cross_edges_i = edges_count(qubit_sequence(i, q)) 
            phase_i = np.exp(-1j*alpha*cross_edges_i)
            phases.append(phase_i)
        
        return np.array(phases)
        



In [46]:
Q2 = QuantumSystem(2)

pp(Q2.Phase,1)


# np.eye(4)


⎡1.0   0     0     0 ⎤
⎢                    ⎥
⎢ 0   -1.0   0     0 ⎥
⎢                    ⎥
⎢ 0    0    -1.0   0 ⎥
⎢                    ⎥
⎣ 0    0     0    1.0⎦
